In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset

import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

import numpy as np 
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

print(torch.__version__)
print(torchvision.__version__)

In [ ]:
x = torch.rand(3, 3)
print(x.device)
print(x.is_cuda)

In [ ]:
train_data = datasets.OxfordIIITPet(
    root="data",
    split="trainval",
    download=True,
    transform=torchvision.transforms.ToTensor(),
    target_transform=None
)

test_data = datasets.OxfordIIITPet(
    root="data",
    split="test",
    download=True,
    transform=torchvision.transforms.ToTensor(),
    target_transform=None
)

In [ ]:
class SRDataset(Dataset):

    def __init__(self, base_data, image_size=(128,128), scale_factor=4, train=True):
        self.dataset = base_data
        self.scale_factor = scale_factor
        self.resize = transforms.Resize((image_size))
        self.crop = transforms.RandomCrop((64,64))
        self.ToTensor = transforms.ToTensor()
        self.inputs = []
        self.targets = []
        print("preprocessing....")
        for img, _ in tqdm(base_data):
            img = transforms.ToPILImage()(img).convert("RGB")
            img = self.resize(img)
            self.train = train
            if self.train:
                img = self.crop(img)
            w, h = img.size
            lr_img = img.resize((w // self.scale_factor, h // self.scale_factor), Image.BICUBIC)
            bicubic_img = lr_img.resize((w, h), Image.BICUBIC)
            self.inputs.append(self.ToTensor(bicubic_img))
            self.targets.append(self.ToTensor(img))
        print("Done!")

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [ ]:
from torch.utils.data import DataLoader

sr_train_data = SRDataset(train_data, image_size=(256,256), scale_factor=2, train=True)
train_dataloader = DataLoader(sr_train_data, batch_size=32, shuffle=True, pin_memory=True)

sr_test_data = SRDataset(test_data, image_size=(256,256), scale_factor=2, train=False)
test_dataloader = DataLoader(sr_test_data, batch_size=16, shuffle=False, pin_memory=True)

In [ ]:
class ResidualBlocks(nn.Module):

    def __init__(self, channels=64):
        super(ResidualBlocks, self).__init__()
        self.layer1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.layer2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        master = x
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        x = x + master
        return x

In [ ]:
class SRCNN(nn.Module):

    def __init__(self):
        super(SRCNN, self).__init__()
        self.layer1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=9, padding=4)
        self.resblocks = nn.Sequential(
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64),
            ResidualBlocks(channels=64)
        )
        self.layer2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5, padding=2)
        self.layer3 = nn.Conv2d(in_channels=32, out_channels=3, kernel_size=5, padding=2)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        x = self.relu(self.layer1(x))
        x = self.resblocks(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x + residual

In [ ]:
# ── Perceptual Loss using VGG19 intermediate features ──────────────────────
# We extract features from relu3_3 (vgg features[:18]) — a common choice
# that captures mid-level texture/structure without being too abstract.
# You can also try [:9] (relu2_2) for lower-level / sharpness focus.

from torchvision.models import VGG19_Weights, vgg19

class VGGPerceptualLoss(nn.Module):
    def __init__(self, feature_layer=18):
        """
        feature_layer options (VGG19 features):
          4  -> relu1_2  (low-level edges)
          9  -> relu2_2  (textures)
          18 -> relu3_3  (mid-level structure)  <-- default, good for SR
          27 -> relu4_3  (semantic content)
        """
        super(VGGPerceptualLoss, self).__init__()
        vgg = vgg19(weights=VGG19_Weights.DEFAULT)
        # Slice the feature extractor up to the chosen layer
        self.feature_extractor = nn.Sequential(
            *list(vgg.features.children())[:feature_layer]
        ).eval()
        # Freeze — we never want to update VGG weights
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # VGG was trained with ImageNet mean/std — normalise inputs
        self.register_buffer(
            'mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            'std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

        self.pixel_loss = nn.L1Loss()
        self.feature_loss = nn.L1Loss()

    def normalize(self, x):
        """Normalize [0,1] tensor to ImageNet stats expected by VGG."""
        return (x - self.mean) / self.std

    def forward(self, output, target, pixel_weight=1.0, perceptual_weight=0.01):
        """
        Combined pixel + perceptual loss.

        pixel_weight      : weight on direct L1 pixel loss (keeps colours stable)
        perceptual_weight : weight on VGG feature loss (sharpens textures)

        Tip: perceptual loss values are typically ~100x larger than pixel loss,
        so 0.01 roughly balances them. Tune to taste.
        """
        # --- Pixel loss ---
        p_loss = self.pixel_loss(output, target)

        # --- Perceptual loss ---
        # Clamp to [0,1] before normalising (model output can overshoot slightly)
        out_feats = self.feature_extractor(self.normalize(output.clamp(0, 1)))
        tgt_feats = self.feature_extractor(self.normalize(target.clamp(0, 1)))
        feat_loss = self.feature_loss(out_feats, tgt_feats)

        return pixel_weight * p_loss + perceptual_weight * feat_loss, p_loss, feat_loss

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SRCNN().to(device)

# Move perceptual loss (and its frozen VGG) to the same device
criterion = VGGPerceptualLoss(feature_layer=18).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epoch = 60
scaler = torch.cuda.amp.GradScaler()

In [ ]:
epoch_loss = []
epoch_pixel_loss = []
epoch_feat_loss = []

for epoch in range(num_epoch):
    model.train()
    running_loss = 0.0
    running_pixel = 0.0
    running_feat = 0.0

    for inputs, targets in train_dataloader:
        inputs  = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(inputs)                          # ✅ SRCNN forward pass
            loss, p_loss, f_loss = criterion(               # ✅ combined loss
                outputs, targets,
                pixel_weight=1.0,
                perceptual_weight=0.01
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss  += loss.item()
        running_pixel += p_loss.item()
        running_feat  += f_loss.item()

    n = len(train_dataloader)
    epoch_loss.append(running_loss / n)
    epoch_pixel_loss.append(running_pixel / n)
    epoch_feat_loss.append(running_feat / n)

    print(f"Epoch [{epoch+1}/{num_epoch}]  "
          f"Total: {running_loss/n:.4f}  "
          f"Pixel: {running_pixel/n:.4f}  "
          f"Feat: {running_feat/n:.4f}")

In [ ]:
# Evaluation loop
model.eval()
running_loss = 0.0

with torch.no_grad():
    for inputs, targets in test_dataloader:
        inputs  = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss, _, _ = criterion(outputs, targets)

        running_loss += loss.item()

    avg_loss = running_loss / len(test_dataloader)
    print(f"Test Loss: {avg_loss:.4f}")

In [ ]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epoch_loss, label="Total")
axes[0].plot(epoch_pixel_loss, label="Pixel (L1)")
axes[0].set_title("Total & Pixel Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epoch_feat_loss, color='orange', label="Perceptual (VGG)")
axes[1].set_title("VGG Perceptual Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visual comparison
idx = 0
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(inputs[idx].float().cpu().permute(1, 2, 0).clamp(0, 1))
axes[0].set_title("Input (bicubic)")

axes[1].imshow(outputs[idx].float().cpu().detach().permute(1, 2, 0).clamp(0, 1))
axes[1].set_title("SRCNN + Perceptual Loss")

axes[2].imshow(targets[idx].float().cpu().permute(1, 2, 0).clamp(0, 1))
axes[2].set_title("Target (original)")

plt.tight_layout()
plt.show()